In [11]:
!pip install PyMuPDF


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [12]:
import fitz  # PyMuPDF
import re
import json
from pathlib import Path
from pprint import pprint
from typing import List, Dict

In [13]:
bns_path = Path.cwd() / "documents" / "BNS.pdf"
bnss_path = Path.cwd() / "documents" / "BNSS.pdf"
output_dir = Path.cwd() / "contents"

In [14]:
def extract_sections_general(pdf_path: str) -> List[Dict]:
    """
    General extractor for legal/numbered sections from a PDF.
    Works for both BNS and BNSS and similar structured documents.
    """

    # --- 1) Read full PDF text by lines ---
    doc = fitz.open(pdf_path)
    lines = []
    for page in doc:
        text = page.get_text("text")
        # Normalize line endings
        text = text.replace("\r\n", "\n").replace("\r", "\n").replace("_", "")
        lines.extend(text.split("\n"))
    doc.close()

    # --- 2) Define regex patterns for section headings ---

    # Pattern A: number and text on same line
    # e.g., "123. Some text starts here" OR "123 ) Some text"
    header_with_text = re.compile(r"^\s*(\d{1,4})\s*(?:\.\s*|\)\s*|\-\s*)(\S.+)$")

    # Pattern B: number on its own line (then text follows on next lines)
    # e.g., "123."
    header_only = re.compile(r"^\s*(\d{1,4})\s*(?:\.\s*|\)\s*|\-\s*)\s*$")

    sections = {}
    current_section = None

    # --- 3) Iterate through all lines ---
    for line in lines:

        # Try pattern A: number + text on same line
        m = header_with_text.match(line)
        if m:
            num = int(m.group(1))
            text = m.group(2).strip()

            current_section = num
            # Start new section content
            sections[current_section] = text
            continue

        # Try pattern B: number only on line
        m2 = header_only.match(line)
        if m2:
            num = int(m2.group(1))
            current_section = num
            sections[current_section] = ""
            continue

        # If currently inside a section, append text
        if current_section is not None:
            # avoid adding blank junk lines
            stripped = line.strip()
            if stripped:
                # accumulate with a space
                sections[current_section] += (
                    " " + stripped if sections[current_section] else stripped
                )

    # --- 4) Convert into sorted list of dicts ---
    result = []
    for num in sorted(sections.keys()):
        # Normalize whitespace
        content = re.sub(r"\s+", " ", sections[num].strip())
        result.append(
            {
                "_id": f"{'bnss' if 'bnss' in pdf_path.lower() else 'bns'}_{num}",
                "section_number": num,
                "content": content,
            }
        )

    return result


def save_as_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [15]:
bns = extract_sections_general(bns_path.as_posix())
save_as_json(bns, output_dir / "bns_sections.json")

In [16]:
bnss = extract_sections_general(bnss_path.as_posix())
save_as_json(bnss, output_dir / "bnss_sections.json")